In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

import numpy as np
import typing as List

https://astra.datastax.com/org/733dce46-269d-4f6d-85c9-be91c6ac8101/database

In [ ]:
ASTRA_DB_API_ENDPOINT = ""
ASTRA_DB_APPLICATION_TOKEN = ""

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6478.22it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
from langchain_astradb import AstraDBVectorStore

vector_store = AstraDBVectorStore(
    collection_name="astra_vector_langchain",
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
)
vector_store

In [12]:

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [13]:
vector_store.add_documents(documents=documents)

['09e3b408d0264f40869a42c8021a164d',
 '4f6bb1e821df49219aad00daadf04f2b',
 'c0b777e9f3af4dbdb97c9a95f35f7c84',
 'd3aa5f6f496f4e3f9e2c7b3791976985',
 'c5fd88cab20a48bf87cde62dbf07fb97',
 '331c15723d674d1ba58de67f7b44a461',
 'f40fd94bd7ae412f883001ef62839db8',
 'c5a25061dfbf4b1e8d278daa39e66843',
 '6bae24a2c665439b947ccd9a06a827d1',
 '0abeb83e57b749918257d2f9e74c63d0']

In [14]:
### Search from Vector store

vector_store.similarity_search("What is the weather")

[Document(id='aa5935a1a99c4279b246c62132c074ed', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='4f6bb1e821df49219aad00daadf04f2b', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='0d5a38a608214b8b9384c7e52ac62c9c', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='37e2ec3145c44314a348b012592caaf8', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.')]

In [15]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res in results:
    print(f'* "{res.page_content}", metadata={res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}


In [16]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res, score in results:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.79] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.79] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.79] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}


In [17]:
### Retriever
retriever=vector_store.as_retriever(
  search_type="mmr",
    search_kwargs={"k": 1},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='d3aa5f6f496f4e3f9e2c7b3791976985', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

### If we have vector sore similary search then why we need retriver

The reason as_retriever() exists is mainly abstraction and integration with RAG chains.

1. Directly using similarity_search()

You can do:

results = vector_store.similarity_search(
    "What is the refund policy?",
    k=4
)

You get back documents:

results
   ↓
[Document, Document, Document, Document]

Then you can manually send those documents to your LLM:

context = "\n".join(doc.page_content for doc in results)

prompt = f"""
Answer the question using this context:

{context}

Question: What is the refund policy?
"""

response = llm.invoke(prompt)

This is perfectly valid RAG.

2. Using a retriever

Instead, you can do:

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

Then:

results = retriever.invoke(
    "What is the refund policy?"
)

At this point, you're still getting documents.

So you might ask:

"Then what's the point?"

The key is that Retriever is an abstraction.

Think of it this way
Without retriever

Your code is tied directly to the vector store:

Question
   ↓
vector_store.similarity_search()
   ↓
Documents
   ↓
LLM
With retriever

You separate "how do I retrieve documents?" from the rest of your RAG pipeline:

                    ┌─ Vector similarity
                    ├─ BM25
Question → Retriever├─ Hybrid search
                    ├─ Multi-query
                    └─ Reranker
                         ↓
                      Documents
                         ↓
                         LLM

That's the major benefit.